notebook_06_embedding.ipynb
Phase 6 — Embedding + ChromaDB

Input:  retrieval_documents_structured_v2.json

         retrieval_documents_full_v2.json

        product_entities_normalized_v2.json  (for metadata)
Output: ChromaDB persisted to Google Drive

        /MyDrive/agromind/chromadb/
            collection_structured  — precise field-level queries
            collection_full        — semantic + Chinese language queries

API calls: OpenAI text-embedding-3-small (~$0.01 total)

In [ ]:
!pip install chromadb openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 915.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [ ]:
import os
import json
import chromadb
from openai import OpenAI
from tqdm import tqdm

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    import getpass
    api_key = getpass.getpass('OpenAI API key: ')

client      = OpenAI(api_key=api_key)
EMBED_MODEL = 'text-embedding-3-large'
print(f'Embedding model: {EMBED_MODEL}')

Embedding model: text-embedding-3-large


In [ ]:
# ── CELL 2 — Mount Google Drive ───────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

CHROMA_PATH = '/content/drive/MyDrive/agromind/chromadb'
os.makedirs(CHROMA_PATH, exist_ok=True)
print(f'ChromaDB path: {CHROMA_PATH}')

Mounted at /content/drive
ChromaDB path: /content/drive/MyDrive/agromind/chromadb


In [ ]:
# ── CELL 3 — Load all input files ────────────────────────────────────────────
from google.colab import files

print('Upload product_entities_normalized_v2.json')
uploaded = files.upload()
with open('product_entities_normalized_v2.json', encoding='utf-8') as f:
    entities = {e['product_id']: e for e in json.load(f)}

print('Upload retrieval_documents_structured_v2.json')
uploaded = files.upload()
with open('retrieval_documents_structured_v2.json', encoding='utf-8') as f:
    structured_docs = json.load(f)

print('Upload retrieval_documents_full_v2.json')
uploaded = files.upload()
with open('retrieval_documents_full_v2.json', encoding='utf-8') as f:
    full_docs = json.load(f)

print(f'\nEntities:   {len(entities)}')
print(f'Structured: {len(structured_docs)}')
print(f'Full:       {len(full_docs)}')

Upload product_entities_normalized_v2.json


Saving product_entities_normalized_v2.json to product_entities_normalized_v2.json
Upload retrieval_documents_structured_v2.json


Saving retrieval_documents_structured_v2.json to retrieval_documents_structured_v2.json
Upload retrieval_documents_full_v2.json


Saving retrieval_documents_full_v2.json to retrieval_documents_full_v2.json

Entities:   114
Structured: 114
Full:       114


In [ ]:
# ── CELL 4 — Embedding function ───────────────────────────────────────────────
def embed_texts(texts, batch_size=50):
    """
    Embed a list of texts using OpenAI text-embedding-3-small.
    Processes in batches to avoid rate limits.
    Returns list of embedding vectors.
    """
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding batches'):
        batch = texts[i:i + batch_size]
        resp  = client.embeddings.create(
            model=EMBED_MODEL,
            input=batch,
        )
        batch_embeddings = [item.embedding for item in resp.data]
        all_embeddings.extend(batch_embeddings)
    return all_embeddings

In [ ]:
# ── CELL 5 — Build metadata for each product ─────────────────────────────────
def build_metadata(entity):
    """
    Build ChromaDB metadata dict from normalized entity.
    Only scalar and short string values — ChromaDB metadata
    does not support nested lists directly.
    Comma-separated strings for list fields enable metadata filtering.
    """
    def safe_join(values):
        if not values:
            return ''
        return ', '.join(str(v) for v in values)

    return {
        'product_id':      entity['product_id'],
        'product_name':    entity['product_name'],
        'product_name_cn': entity['product_name_cn'],
        'product_types':   safe_join(entity.get('product_types', [])),
        'target_crops':    safe_join(entity.get('target_crops', [])[:10]),  # cap at 10
        'target_diseases': safe_join(entity.get('target_diseases', [])[:10]),
        'target_pests':    safe_join(entity.get('target_pests', [])[:10]),
        'active_ingredients': safe_join(entity.get('active_ingredients', [])[:5]),
        'has_diseases':    len(entity.get('target_diseases', [])) > 0,
        'has_pests':       len(entity.get('target_pests', [])) > 0,
        'has_ingredients': len(entity.get('active_ingredients', [])) > 0,
        'is_pesticide':    any(t in ['Pesticide', 'Insecticide', 'Fungicide',
                                      'Herbicide', 'Acaricide']
                               for t in entity.get('product_types', [])),
        'is_microbial':    'Microbial Agent' in entity.get('product_types', []),
        'is_fertilizer':   any('Fertilizer' in t
                               for t in entity.get('product_types', [])),
    }


In [ ]:
# ── CELL 6 — Test embedding before full run ───────────────────────────────────
# Test on 3 documents first — verify embedding works and check dimensions
print('Testing embedding on 3 documents...')
test_texts = [structured_docs[0]['document'],
              structured_docs[1]['document'],
              structured_docs[2]['document']]
test_embeds = embed_texts(test_texts, batch_size=3)
print(f'Embedding dimension: {len(test_embeds[0])}')  # should be 1536
print(f'Test embeddings: {len(test_embeds)} ✓')
print(f'Sample metadata: {build_metadata(entities["AF0001"])}')

Testing embedding on 3 documents...


Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.82s/it]

Embedding dimension: 3072
Test embeddings: 3 ✓
Sample metadata: {'product_id': 'AF0001', 'product_name': 'Citrus Junliqing', 'product_name_cn': '柑橘菌立清', 'product_types': 'Growth Promoter, Microbial Agent, Microbial Fertilizer', 'target_crops': 'Chili Pepper, Citrus, Vegetables, Fruit Trees, Melons, Field Crops, Medicinal Herbs, Flowers, Seedlings, Tea Trees', 'target_diseases': 'Downy Mildew, Gray Mold, Purple Spot, Anthracnose, Leaf Spot, Wilt, Root Rot, Damping-off, Soft Rot, Ginger Wilt', 'target_pests': '', 'active_ingredients': 'Bacillus subtilis, Bacillus licheniformis, Bacillus pumilus, Bacillus gelatinous, Bacillus amyloliquefaciens', 'has_diseases': True, 'has_pests': False, 'has_ingredients': True, 'is_pesticide': False, 'is_microbial': True, 'is_fertilizer': True}


In [ ]:
# ── CELL 7 — Initialize ChromaDB ──────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete existing collections if re-running — prevents dimension mismatch errors
for col_name in ['collection_structured', 'collection_full']:
    try:
        chroma_client.delete_collection(col_name)
        print(f'Deleted existing {col_name}')
    except Exception:
        pass  # collection didn't exist yet

# Create fresh collections
col_structured = chroma_client.create_collection(
    name='collection_structured',
    metadata={'description': 'Structured product summaries — precise field queries',
              'embed_model': EMBED_MODEL,
              'source': 'retrieval_documents_structured_v2.json'}
)

col_full = chroma_client.create_collection(
    name='collection_full',
    metadata={'description': 'Full bilingual documents — semantic + Chinese queries',
              'embed_model': EMBED_MODEL,
              'source': 'retrieval_documents_full_v2.json'}
)

print(f'Created: collection_structured')
print(f'Created: collection_full')

Created: collection_structured
Created: collection_full


In [ ]:
# In notebook_06 — add without deleting existing
col_structured.add(ids=[new_pid], documents=[new_doc],
                   embeddings=[new_embed], metadatas=[new_meta])

In [ ]:
# ── CELL 8 — Embed and store STRUCTURED documents ─────────────────────────────
print('Embedding structured documents...')

s_ids       = [d['product_id'] for d in structured_docs]
s_texts     = [d['document']   for d in structured_docs]
s_metadatas = [build_metadata(entities[d['product_id']]) for d in structured_docs]

s_embeddings = embed_texts(s_texts)

col_structured.add(
    ids=s_ids,
    documents=s_texts,
    embeddings=s_embeddings,
    metadatas=s_metadatas,
)

print(f'Stored {col_structured.count()} documents in collection_structured ✓')


Embedding structured documents...


Embedding batches: 100%|██████████| 3/3 [00:01<00:00,  2.05it/s]


Stored 114 documents in collection_structured ✓


In [ ]:
# ── CELL 9 — Embed and store FULL documents ───────────────────────────────────
print('Embedding full documents...')

f_ids       = [d['product_id'] for d in full_docs]
f_texts     = [d['document']   for d in full_docs]
f_metadatas = [build_metadata(entities[d['product_id']]) for d in full_docs]

f_embeddings = embed_texts(f_texts)

col_full.add(
    ids=f_ids,
    documents=f_texts,
    embeddings=f_embeddings,
    metadatas=f_metadatas,
)

print(f'Stored {col_full.count()} documents in collection_full ✓')


Embedding full documents...


Embedding batches: 100%|██████████| 3/3 [00:01<00:00,  2.23it/s]


Stored 114 documents in collection_full ✓


In [ ]:
# ── CELL 10 — Verify storage ──────────────────────────────────────────────────
print('=== CHROMADB VERIFICATION ===')
print(f'collection_structured: {col_structured.count()} documents')
print(f'collection_full:       {col_full.count()} documents')

# Peek at first 3 entries
peek = col_structured.peek(limit=3)
print(f'\nSample IDs: {peek["ids"]}')
print(f'Sample metadata keys: {list(peek["metadatas"][0].keys())}')


=== CHROMADB VERIFICATION ===
collection_structured: 114 documents
collection_full:       114 documents

Sample IDs: ['AF0001', 'AF0002', 'AF0003']
Sample metadata keys: ['product_types', 'target_crops', 'has_pests', 'product_id', 'product_name_cn', 'has_ingredients', 'target_diseases', 'target_pests', 'product_name', 'is_fertilizer', 'is_pesticide', 'has_diseases', 'active_ingredients', 'is_microbial']


In [ ]:
# ── CELL 11 — Smoke test: 5 real queries ─────────────────────────────────────
# Test queries covering English, Chinese, disease, pest, dosage, product name
TEST_QUERIES = [
    'citrus yellowing leaves treatment',           # English symptom query
    '柑橘叶片发黄怎么处理',                         # Chinese symptom query
    'what treats root rot in tomatoes',            # English disease query
    'Bacillus subtilis microbial agent',           # ingredient query
    'dosage for 1000 times dilution spray',        # dosage query
]

print('=== SMOKE TEST — 5 QUERIES ===\n')
for query in TEST_QUERIES:
    # Embed the query
    q_embed = client.embeddings.create(model=EMBED_MODEL, input=[query])
    q_vector = q_embed.data[0].embedding

    # Search both collections
    results_s = col_structured.query(
        query_embeddings=[q_vector],
        n_results=2,
        include=['documents', 'metadatas', 'distances']
    )
    results_f = col_full.query(
        query_embeddings=[q_vector],
        n_results=2,
        include=['documents', 'metadatas', 'distances']
    )

    print(f'Query: "{query}"')
    print(f'  Structured top-1: {results_s["ids"][0][0]} '
          f'({results_s["metadatas"][0][0]["product_name"]}) '
          f'dist={results_s["distances"][0][0]:.3f}')
    print(f'  Full      top-1: {results_f["ids"][0][0]} '
          f'({results_f["metadatas"][0][0]["product_name"]}) '
          f'dist={results_f["distances"][0][0]:.3f}')
    print()


=== SMOKE TEST — 5 QUERIES ===

Query: "citrus yellowing leaves treatment"
  Structured top-1: AF0001 (Citrus Junliqing) dist=1.010
  Full      top-1: AF0013 (Water-Soluble Fertilizer Specially for Citrus) dist=1.073

Query: "柑橘叶片发黄怎么处理"
  Structured top-1: AF0013 (Water-Soluble Fertilizer Specially for Citrus) dist=1.044
  Full      top-1: AF0013 (Water-Soluble Fertilizer Specially for Citrus) dist=1.001

Query: "what treats root rot in tomatoes"
  Structured top-1: AF0024 (Rooting Solution) dist=1.004
  Full      top-1: AF0024 (Rooting Solution) dist=1.101

Query: "Bacillus subtilis microbial agent"
  Structured top-1: PN0001 (Bacillus thuringiensis 4000 IU/μL) dist=1.016
  Full      top-1: PN0001 (Bacillus thuringiensis 4000 IU/μL) dist=1.015

Query: "dosage for 1000 times dilution spray"
  Structured top-1: PN0001 (Bacillus thuringiensis 4000 IU/μL) dist=1.137
  Full      top-1: PN0001 (Bacillus thuringiensis 4000 IU/μL) dist=1.201



In [ ]:
# ── CELL 12 — Save embedding log ─────────────────────────────────────────────
embedding_log = {
    'embed_model':          EMBED_MODEL,
    'embed_dimensions':     len(s_embeddings[0]),
    'collection_structured': {
        'count': col_structured.count(),
        'path':  CHROMA_PATH,
    },
    'collection_full': {
        'count': col_full.count(),
        'path':  CHROMA_PATH,
    },
    'chroma_path': CHROMA_PATH,
}

with open('embedding_log_v2.json', 'w', encoding='utf-8') as f:
    json.dump(embedding_log, f, indent=2)

files.download('embedding_log_v2.json')
print('Saved → embedding_log_v2.json')

print(f"""
=== PHASE 6 COMPLETE ✓ ===
Model:   {EMBED_MODEL}
Dims:    {len(s_embeddings[0])}
Stored:  {col_structured.count()} structured + {col_full.count()} full
Path:    {CHROMA_PATH}
Next:    notebook_07_retrieval_testing.ipynb
""")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved → embedding_log_v2.json

=== PHASE 6 COMPLETE ✓ ===
Model:   text-embedding-3-large
Dims:    3072
Stored:  114 structured + 114 full
Path:    /content/drive/MyDrive/agromind/chromadb
Next:    notebook_07_retrieval_testing.ipynb

